# ⚡ Transvid: Neural Latent Media Compression Machine

Welcome to the official **Transvid** demonstration notebook!

This notebook demonstrates how Transvid encodes audio and video into **compact neural latent representations** (`.tvid` format) and reconstructs high-fidelity media back using deep neural decoders.

---

### Step 1: Verify Cloud GPU and Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Install Dependencies

In [ ]:
# Install dependencies directly
!pip install -q encodec torchaudio soundfile opencv-python-headless numpy zstandard tqdm

## 🎵 Part 1: Audio Latent Compression (400x+ Smaller)

In [ ]:
import numpy as np
import torchaudio
import torch

# Create a 5-second musical sample (C-Major chord)
sample_rate = 44100
duration = 5.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
chord = (
    0.4 * np.sin(2 * np.pi * 261.63 * t) +
    0.3 * np.sin(2 * np.pi * 329.63 * t) +
    0.3 * np.sin(2 * np.pi * 392.00 * t)
).astype(np.float32)

torchaudio.save("input_sample.wav", torch.from_numpy(chord).unsqueeze(0), sample_rate)

# Compress to .tvid
!python -m src.cli compress input_sample.wav -o audio_sample.tvid --bandwidth 6.0
!python -m src.cli info audio_sample.tvid

# Decompress back to WAV
!python -m src.cli decompress audio_sample.tvid -o audio_restored.wav

In [ ]:
import os
from IPython.display import Audio, display, HTML

orig_size = os.path.getsize("input_sample.wav")
tvid_size = os.path.getsize("audio_sample.tvid")
ratio = orig_size / tvid_size

display(HTML(f"""
<h3>Audio Compression Summary:</h3>
<ul>
  <li><b>Original WAV:</b> {orig_size / 1024:.1f} KB</li>
  <li><b>Transvid (.tvid):</b> {tvid_size / 1024:.1f} KB</li>
  <li><b>Savings:</b> <span style='color:green;'><b>{ratio:.1f}x smaller!</b></span> ({100 - (tvid_size/orig_size*100):.1f}% reduction)</li>
</ul>
"""))

print("Original Audio:")
display(Audio("input_sample.wav"))

print("Reconstructed Audio from .tvid:")
display(Audio("audio_restored.wav"))

## 🎬 Part 2: Video Latent Compression (Spatio-Temporal Autoencoding)

In [ ]:
import cv2

# Generate a synthetic test video (60 frames, moving circle, 128x128 @ 30 FPS)
w, h, fps, num_frames = 128, 128, 30.0, 60
out = cv2.VideoWriter("input_video.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
for i in range(num_frames):
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[:, :] = [i * 4 % 255, 120, 160]
    cx = int(w/2 + 35 * np.cos(i * 0.15))
    cy = int(h/2 + 35 * np.sin(i * 0.15))
    cv2.circle(frame, (cx, cy), 18, (255, 255, 255), -1)
    cv2.putText(frame, f"Frame {i}", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    out.write(frame)
out.release()
print("Created input_video.mp4")

# Compress video to .tvid
!python -m src.cli compress input_video.mp4 -o video_sample.tvid --gop 16
!python -m src.cli info video_sample.tvid

# Decompress back to playable MP4
!python -m src.cli decompress video_sample.tvid -o video_restored.mp4

In [ ]:
v_orig_size = os.path.getsize("input_video.mp4")
v_tvid_size = os.path.getsize("video_sample.tvid")
v_ratio = v_orig_size / v_tvid_size

display(HTML(f"""
<h3>Video Compression Summary:</h3>
<ul>
  <li><b>Original MP4:</b> {v_orig_size / 1024:.1f} KB</li>
  <li><b>Transvid (.tvid):</b> {v_tvid_size / 1024:.1f} KB</li>
  <li><b>Compression:</b> <span style='color:green;'><b>{v_ratio:.1f}x reduction</b></span></li>
</ul>
"""))